# Base Checks: Трек 7 — Сглаживание Нестерова и Проксимальные методы
Проверка корректности реализаций оракулов и методов оптимизации.

In [1]:
import sys
import os
os.chdir('..')

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from src.oracles import (
    L1RegOracle, RegressionSmoothOracle, ClassificationSmoothOracle,
    RegressionNonsmoothOracle, ClassificationNonsmoothOracle,
    RegressionProxOracle, ClassificationProxOracle,
    BarrierL1Oracle, QuadraticOracle,
    grad_finite_diff, hess_finite_diff
)
from src.optimization import (
    subgradient_method, proximal_gradient_method,
    proximal_fast_gradient_method, frank_wolfe_method, barrier_method
)
from src.utils import load_datasets, load_space_ga, load_phishing, create_oracle_functions

## 1. Проверка L1RegOracle: prox и lmo

In [3]:
h = L1RegOracle(regcoef=0.5)

x = np.array([1.5, -2.0, 0.3, -0.1])
alpha = 1.0
prox_x = h.prox(x, alpha)
expected = np.array([1.0, -1.5, 0.0, 0.0])  # soft threshold with threshold = 0.5
print('prox result:  ', prox_x)
print('expected:     ', expected)
assert np.allclose(prox_x, expected), 'prox FAILED'
print('prox: OK')

grad = np.array([0.1, -0.8, 0.3, -0.2])
R = 2.0
y = h.lmo(grad, R)
print('lmo result:   ', y)  # should be R*e_1 in -sign direction of largest |grad|
assert np.allclose(y, [0, 2.0, 0, 0]), 'lmo FAILED'
print('lmo: OK')

prox result:   [ 1.  -1.5  0.  -0. ]
expected:      [ 1.  -1.5  0.   0. ]
prox: OK
lmo result:    [0. 2. 0. 0.]
lmo: OK


## 2. Проверка BarrierL1Oracle: градиент и гессиан (конечные разности)

In [4]:
A = np.array([[3.0, 1.0], [1.0, 2.0]])
b_vec = np.array([1.0, 1.0])
smooth = QuadraticOracle(A, b_vec)

barrier = BarrierL1Oracle(smooth_oracle=smooth, lambda_reg=0.5, t=1.0)

# Strictly feasible point: u_i > |x_i|
z0 = np.array([0.2, -0.3, 1.0, 1.0])  # x=[0.2,-0.3], u=[1.0,1.0]

grad_analytic = barrier.grad(z0)
grad_fd = grad_finite_diff(barrier.func, z0, eps=1e-7)
print('Gradient error (analytic vs FD):', np.max(np.abs(grad_analytic - grad_fd)))
assert np.allclose(grad_analytic, grad_fd, atol=1e-5), 'BarrierL1Oracle gradient FAILED'
print('BarrierL1Oracle gradient: OK')

hess_analytic = barrier.hess(z0)
hess_fd = hess_finite_diff(barrier.func, z0, eps=1e-5)
print('Hessian error (analytic vs FD):', np.max(np.abs(hess_analytic - hess_fd)))
assert np.allclose(hess_analytic, hess_fd, atol=1e-4), 'BarrierL1Oracle hessian FAILED'
print('BarrierL1Oracle hessian: OK')

Gradient error (analytic vs FD): 2.652148812787303e-07
BarrierL1Oracle gradient: OK
Hessian error (analytic vs FD): 6.916649696986354e-05
BarrierL1Oracle hessian: OK


## 3. Проверка субградиентного метода

In [5]:
from src.oracles import BaseNonsmoothConvexOracle

class AbsOracle(BaseNonsmoothConvexOracle):
    """f(x) = sum |x_i|, minimum at x=0."""
    def func(self, x): return np.sum(np.abs(x))
    def subgrad(self, x): return np.sign(x)

oracle = AbsOracle()
x0 = np.array([3.0, -4.0])
x_star, msg, hist = subgradient_method(oracle, x0, tolerance=1e-4, max_iter=10000, alpha_0=1.0, trace=True)
print(f'subgradient: msg={msg}, ||x*||={np.linalg.norm(x_star):.2e}, f*={oracle.func(x_star):.2e}')
print(f'iterations: {len(hist["func"])}')

subgradient: msg=success, ||x*||=0.00e+00, f*=0.00e+00
iterations: 5234


## 4. Проверка ISTA и FISTA

In [6]:
from src.oracles import BaseCompositeOracle, BaseSmoothOracle

n = 10
np.random.seed(42)
A_mat = np.random.randn(n, n)
A_mat = A_mat.T @ A_mat + np.eye(n)  # PD
b_true = np.random.randn(n)

class QuadSmooth(BaseSmoothOracle):
    def func(self, x): return 0.5 * x @ A_mat @ x - b_true @ x
    def grad(self, x): return A_mat @ x - b_true
    def hess(self, x): return A_mat

regcoef = 0.1
f = QuadSmooth()
h = L1RegOracle(regcoef)
comp = BaseCompositeOracle(f, h)

x0 = np.zeros(n)
x_ista, msg_ista, hist_ista = proximal_gradient_method(comp, x0, L_0=1.0, tolerance=1e-8, max_iter=2000, trace=True)
x_fista, msg_fista, hist_fista = proximal_fast_gradient_method(comp, x0, L_0=1.0, tolerance=1e-8, max_iter=2000, trace=True)

print(f'ISTA:  msg={msg_ista}, iters={len(hist_ista["func"])-1}, f*={comp.func(x_ista):.6f}')
print(f'FISTA: msg={msg_fista}, iters={len(hist_fista["func"])-1}, f*={comp.func(x_fista):.6f}')
assert abs(comp.func(x_ista) - comp.func(x_fista)) < 1e-4, 'ISTA and FISTA converge to different values'
print('ISTA и FISTA сошлись к одному решению: OK')

ISTA:  msg=success, iters=228, f*=-0.829815
FISTA: msg=success, iters=377, f*=-0.829815
ISTA и FISTA сошлись к одному решению: OK


## 5. Проверка метода Франка-Вулфа

In [7]:
# Minimize quadratic over L1-ball
class QuadWithLMO(BaseSmoothOracle):
    def __init__(self, A, b):
        self._A = A
        self._b = b
        self._lmo_oracle = L1RegOracle(1.0)
    def func(self, x): return 0.5 * x @ self._A @ x - self._b @ x
    def grad(self, x): return self._A @ x - self._b
    def lmo(self, grad, R): return self._lmo_oracle.lmo(grad, R)

fw_oracle = QuadWithLMO(A_mat, b_true)
x0 = np.zeros(n)
R = 2.0
x_fw, msg_fw, hist_fw = frank_wolfe_method(
    fw_oracle, x0, R=R, tolerance=1e-4, max_iter=2000, trace=True)

print(f'FW standard: msg={msg_fw}, iters={len(hist_fw["func"])}, f*={fw_oracle.func(x_fw):.6f}')
print(f'||x*||_1 = {np.linalg.norm(x_fw, 1):.4f} <= R={R}: {np.linalg.norm(x_fw, 1) <= R + 1e-6}')

x_fw_arm, msg_arm, hist_arm = frank_wolfe_method(
    fw_oracle, x0, R=R, tolerance=1e-4, max_iter=2000, step_size_strategy='armijo', trace=True)
print(f'FW armijo:   msg={msg_arm}, iters={len(hist_arm["func"])}, f*={fw_oracle.func(x_fw_arm):.6f}')

FW standard: msg=iterations_exceeded, iters=2000, f*=-1.011134
||x*||_1 = 1.9996 <= R=2.0: True
FW armijo:   msg=iterations_exceeded, iters=2000, f*=-1.008374


## 6. Проверка барьерного метода

In [8]:
# Minimize small quadratic + L1 via barrier
n_small = 5
A_small = 2.0 * np.eye(n_small)
b_small = np.array([1.0, 2.0, 3.0, 0.5, -1.0])
smooth_small = QuadraticOracle(A_small, b_small)

lambda_reg = 0.5
x0_bar = np.zeros(n_small)
u0_bar = np.ones(n_small) * 1.5  # u_i > |x_i| = 0

x_bar, u_bar, msg_bar, hist_bar = barrier_method(
    smooth_small, x0_bar, u0_bar, lambda_reg,
    t_0=1.0, mu=10.0,
    tolerance_inner=1e-8, tolerance_outer=1e-5,
    max_iter=50, max_inner_iter=100,
    trace=True
)

print(f'barrier: msg={msg_bar}')
print(f'x* = {x_bar}')
print(f'||x*||_1 = {np.linalg.norm(x_bar, 1):.4f}')

# Compare with ISTA result on the same problem
comp_small = BaseCompositeOracle(smooth_small, L1RegOracle(lambda_reg))
x_ista_small, _, _ = proximal_gradient_method(comp_small, x0_bar, L_0=1.0, tolerance=1e-8, max_iter=5000)
print(f'ISTA x* = {x_ista_small}')
print(f'Разница между barrier и ISTA: {np.linalg.norm(x_bar - x_ista_small):.2e}')

barrier: msg=success
x* = [ 0.25002     0.75000667  1.250004    0.00223105 -0.25002   ]
||x*||_1 = 2.5023
ISTA x* = [ 0.25  0.75  1.25  0.   -0.25]
Разница между barrier и ISTA: 2.23e-03


## 7. Проверка оракулов регрессии и классификации

In [9]:
load_datasets('data')

X_reg, y_reg = load_space_ga('data')
m, n_reg = X_reg.shape
print(f'space_ga: m={m}, n={n_reg}')

matvec_Ax, matvec_ATx, matmat_ATsA = create_oracle_functions(X_reg)
reg_oracle = RegressionSmoothOracle(matvec_Ax, matvec_ATx, matmat_ATsA, y_reg, regcoef=1.0/m)

x_test = np.zeros(n_reg)
grad_analytic = reg_oracle.grad(x_test)
grad_fd = grad_finite_diff(reg_oracle.func, x_test)
err = np.linalg.norm(grad_analytic - grad_fd) / max(np.linalg.norm(grad_analytic), 1e-10)
print(f'RegressionSmoothOracle grad error (relative): {err:.2e}')
assert err < 1e-4, 'RegressionSmoothOracle gradient check FAILED'
print('RegressionSmoothOracle: OK')

X_cls, y_cls = load_phishing('data')
m_cls, n_cls = X_cls.shape
print(f'phishing: m={m_cls}, n={n_cls}')

matvec_Ax_c, matvec_ATx_c, matmat_ATsA_c = create_oracle_functions(X_cls)
cls_oracle = ClassificationSmoothOracle(matvec_Ax_c, matvec_ATx_c, matmat_ATsA_c, y_cls, regcoef=1.0/m_cls)

x_test_c = np.zeros(n_cls)
grad_analytic_c = cls_oracle.grad(x_test_c)
grad_fd_c = grad_finite_diff(cls_oracle.func, x_test_c)
err_c = np.linalg.norm(grad_analytic_c - grad_fd_c) / max(np.linalg.norm(grad_analytic_c), 1e-10)
print(f'ClassificationSmoothOracle grad error (relative): {err_c:.2e}')
assert err_c < 1e-4, 'ClassificationSmoothOracle gradient check FAILED'
print('ClassificationSmoothOracle: OK')

space_ga: m=3107, n=6
RegressionSmoothOracle grad error (relative): 1.54e-07
RegressionSmoothOracle: OK
phishing: m=11055, n=68
ClassificationSmoothOracle grad error (relative): 8.41e-08
ClassificationSmoothOracle: OK
